# A Tour of the Exponential Families

A generalized linear model (GLM) describes a response y in two parts:

- a **family** of distributions for y given the predictors X, such as the Gaussian, Bernoulli, or Poisson, and
- a **link function** g relating the mean of that distribution to a linear function of the predictors: `g(E[y | X]) = X @ β`.

The family says what *kind* of response we have (continuous, binary, a count, a positive amount); the link keeps the conditional mean in the range that kind of response allows. This notebook fits a model from each of `py-glm`'s families to data simulated from that very model, so we can compare every estimate with the truth.

In [1]:
import numpy as np
import pandas as pd

from glm.datasets import make_frame
from glm.families import Bernoulli, Exponential, Gamma, Gaussian, Poisson, QuasiPoisson
from glm.glm import GLM

Each simulation below gets its own seeded random number generator, so that it produces the same data however the notebook's cells are run.

`make_frame` simulates a data frame from a GLM, and returns it along with the formula and true coefficients that generated it. This helper lines a fit model's estimates up against the truth: the z column is how many standard errors each estimate lies from its true value, so for a correctly specified model these should look like draws from a standard normal, mostly between -2 and 2.

In [2]:
def compare(model, sim):
    """A fit model's estimates next to the true coefficients."""
    table = pd.DataFrame(
        {
            "estimate": model.coef_,
            "std error": model.coef_standard_error_,
            "truth": sim.coef,
        },
        index=model.X_names,
    )
    table["z"] = (table["estimate"] - table["truth"]) / table["std error"]
    return table.round(3)

## Gaussian: linear regression

The Gaussian family, which uses the identity link, is classical linear regression: `E[y | X] = X @ β`, with constant variance `σ²`. Here `σ² = 4`.

In [3]:
sim = make_frame(
    Gaussian(),
    n_samples=1000,
    n_uncorrelated=3,
    n_correlated=3,
    dispersion=4.0,
    correlation=0.8,
    rng=np.random.default_rng(1),
)
sim.data.head()

,x1,x2,x3,x4,x5,x6,y
0,0.345584,0.821618,0.330437,0.497470,0.658231,1.114760,4.216196
1,-1.303157,0.905356,0.446375,-0.183169,-0.675443,-0.880380,5.969742
2,-0.536953,0.581118,0.364572,0.798780,0.711332,0.866167,2.967080
3,0.294132,0.028422,0.546713,-0.836799,-0.564695,-0.478946,-2.594374
4,-0.736454,-0.162910,-0.482119,0.597599,-0.383956,-0.295439,1.301791


In [4]:
linear_model = GLM(Gaussian()).fit(sim.data, formula=sim.formula)
compare(linear_model, sim)

,estimate,std error,truth,z
Intercept,-0.338,0.061,-0.371,0.544
x1,-0.772,0.061,-0.782,0.160
x2,0.827,0.063,0.795,0.497
x3,-0.244,0.060,-0.169,-1.249
x4,0.386,0.117,0.243,1.226
x5,-0.408,0.111,-0.359,-0.443
x6,0.204,0.113,0.245,-0.355


The standard errors of `x4`, `x5`, and `x6` are larger than the others. Those three predictors were simulated with a correlation of 0.8 between each pair, and when predictors move together, the data can't easily inform of their seperate effects. The *variance inflation factor* (VIF) measures how much: it's the factor by which a coefficient's variance grows because of its predictor's correlation with the others, and it's the diagonal of the inverse of the predictors correlation matrix. Its square root is the factor by which the standard error grows:

In [5]:
predictors = sim.data[["x1", "x2", "x3", "x4", "x5", "x6"]]
vif = np.diag(np.linalg.inv(predictors.corr()))
pd.DataFrame({"VIF": vif, "√VIF": np.sqrt(vif)}, index=predictors.columns).round(2)

,VIF,√VIF
x1,1.01,1.00
x2,1.01,1.00
x3,1.01,1.00
x4,3.68,1.92
x5,3.32,1.82
x6,3.49,1.87


For three predictors with pairwise correlation ρ = 0.8, the VIF is `(1 + ρ) / ((1 - ρ)(1 + 2ρ)) ≈ 3.46`, so their standard errors are about `√3.46 ≈ 1.86` times those of the uncorrelated predictors, as in the table above.

#### Aside: Where does that formula come from? 

The [variance inflation factor](https://en.wikipedia.org/wiki/Variance_inflation_factor) of predictor `j` is `1 / (1 - R_j²)`, where `R_j²` is the R² from regressing predictor `j` on the other predictors. That's the same as the j-th diagonal entry of `C⁻¹`, the inverse of the predictors' correlation matrix `C`: by the [block-matrix inverse formula](https://en.wikipedia.org/wiki/Block_matrix#Inversion), that entry is one over the [Schur complement](https://en.wikipedia.org/wiki/Schur_complement) of the other predictors' block, and on the correlation scale that Schur complement is `1 - R_j²`, the share of predictor j's variance the others can't explain.

For k predictors with every pairwise correlation equal to ρ, the correlation matrix is `C = (1 - ρ) I + ρ 1 1ᵀ`, a multiple of the identity plus a rank one matrix. The [Sherman–Morrison formula](https://en.wikipedia.org/wiki/Sherman%E2%80%93Morrison_formula) inverts it in closed form, and its diagonal is `(1 + (k - 2)ρ) / ((1 - ρ)(1 + (k - 1)ρ))`, which for k = 3 is the formula above.

The Gaussian family has a *dispersion* parameter, its variance `σ²`, which `py-glm` estimates after fitting, and which scales the standard errors. Its estimate is within about two of its own standard errors, `σ² √(2 / n) ≈ 0.18`, of the true value of 4:

In [6]:
print(f"σ² = {linear_model.dispersion_:2.2f}")

σ² = 3.69


## Bernoulli: logistic regression

For a binary response, the Bernoulli family models the probability that `y = 1`. The logit link, `g(p) = log(p / (1 - p))`, maps any value of `X @ β` to a probability between 0 and 1.

In [7]:
sim = make_frame(
    Bernoulli(),
    n_samples=2000,
    n_uncorrelated=3,
    n_correlated=3,
    correlation=0.8,
    rng=np.random.default_rng(2),
)
sim.data.head()

,x1,x2,x3,x4,x5,x6,y
0,0.189053,-0.522748,-0.413064,-1.263850,-0.792247,-0.300193,1
1,-2.441467,1.799707,1.144166,1.354889,1.819098,1.085331,1
2,-0.325423,0.773807,0.281211,-0.277463,-0.042344,-0.143909,0
3,-0.553823,0.977567,-0.310557,-0.445398,-1.757402,-1.294210,1
4,-0.328824,-0.792147,0.454958,0.402211,0.263317,1.526337,0


In [8]:
logistic_model = GLM(Bernoulli()).fit(sim.data, formula=sim.formula)
compare(logistic_model, sim)

,estimate,std error,truth,z
Intercept,-0.267,0.049,-0.204,-1.272
x1,-0.018,0.049,0.053,-1.435
x2,0.774,0.055,0.851,-1.414
x3,-0.345,0.050,-0.250,-1.889
x4,-0.009,0.092,-0.011,0.017
x5,0.127,0.093,0.067,0.642
x6,-0.396,0.094,-0.304,-0.990


`predict` returns the estimated conditional mean `E[y | X]`, which for a binary response is a *probability*, not a class: thresholding at `0.5` (or anything else) is up to you. Here are the predictions next to the true probabilities that generated each response:

In [9]:
pd.DataFrame(
    {
        "true probability": sim.mean,
        "predicted probability": logistic_model.predict(sim.data),
        "y": sim.data["y"],
    }
).head().round(3)

,true probability,predicted probability,y
0,0.381,0.377,1
1,0.666,0.637,1
2,0.601,0.573,0
3,0.723,0.711,1
4,0.188,0.167,0


A Bernoulli response has no dispersion parameter: its variance, `p(1 - p)`, is fixed by its mean. So `dispersion_` is always 1.

In [10]:
logistic_model.dispersion_

1.0

## Poisson: counts, and exposures

The Poisson family models conditional counts. It uses the log link, so that `E[y | X] = exp(X @ β)` is always positive, and each predictor acts *multiplicatively* on the mean.

In observational data counts often accumulate over different *exposures*: claims over policies held for different lengths of time, or event counts in regions of different areas or volumes. The mean is assumed proportional to the exposure, `E[y | X] = exposure * exp(X @ β)`, and taking logs, `log(exposure)` enters the linear predictor with a coefficient fixed at one: an **offset**. `make_frame` simulates such data with `exposure_range`, and returns the offset to use.

In [11]:
sim = make_frame(
    Poisson(),
    n_samples=2000,
    n_uncorrelated=3,
    n_correlated=3,
    coef_range=(-0.5, 0.5),
    exposure_range=(1.0, 5.0),
    correlation=0.8,
    rng=np.random.default_rng(5),
)
sim.data.head()

,x1,x2,x3,x4,x5,x6,exposure,y
0,-0.801931,-1.324359,-0.248362,-2.774951,-2.181598,-2.183055,2.062054,2
1,0.420445,1.136047,0.109706,2.194172,1.148832,1.420497,2.940802,0
2,-0.552647,-0.784780,0.748746,1.622907,1.189039,1.764926,3.794261,3
3,1.634783,0.272769,-1.233329,0.260814,0.186674,0.566557,1.487730,2
4,-0.958265,1.600019,0.202882,-0.371399,-0.813772,-0.238339,1.922718,2


In [12]:
poisson_model = GLM(Poisson()).fit(sim.data, formula=sim.formula, offset=sim.offset)
compare(poisson_model, sim)

,estimate,std error,truth,z
Intercept,-0.433,0.017,-0.434,0.073
x1,0.271,0.016,0.280,-0.612
x2,0.101,0.015,0.104,-0.154
x3,-0.267,0.015,-0.272,0.362
x4,-0.065,0.028,-0.085,0.746
x5,-0.499,0.028,-0.489,-0.354
x6,0.371,0.028,0.402,-1.125


The coefficients are now effects on the *rate* of counts per unit of exposure. Forgetting the offset leaves the model to explain the exposure's effect with the predictors it has. These predictors are independent of the exposure, so their estimates survive, but the intercept changes, and, as we'll see, the standard errors are no longer trustworthy:

In [13]:
no_offset_model = GLM(Poisson()).fit(sim.data, formula=sim.formula)
compare(no_offset_model, sim)

,estimate,std error,truth,z
Intercept,0.666,0.017,-0.434,65.000
x1,0.267,0.015,0.280,-0.825
x2,0.088,0.015,0.104,-1.085
x3,-0.271,0.015,-0.272,0.064
x4,-0.045,0.028,-0.085,1.443
x5,-0.535,0.028,-0.489,-1.643
x6,0.379,0.028,0.402,-0.830


In [14]:
print(f"Intercept without the offset: {no_offset_model.coef_[0]:.3f}")
print(f"Intercept with the offset: {poisson_model.coef_[0]:.3f}")
print(f"True Intercept: {sim.coef['Intercept']:.3f}")

print(f"True intercept + log of the average exposure: {sim.coef['Intercept'] + np.log(sim.data['exposure'].mean()):.3f}")
print(f"True intercept + average of the log exposures: {sim.coef['Intercept'] + np.mean(sim.offset):.3f}")

Intercept without the offset: 0.666
Intercept with the offset: -0.433
True Intercept: -0.434
True intercept + log of the average exposure: 0.664
True intercept + average of the log exposures: 0.578


Which one does the intercept absorb? Since the exposure is independent of the predictors, the mean count given the predictors alone is `E[y | X] = E[exposure] exp(X @ β)`, so the intercept becomes the true intercept plus the log of the *average exposure*. That's not the average of the log exposures, which is smaller: log is concave, so averaging before or after taking it gives different answers ([Jensen's inequality](https://en.wikipedia.org/wiki/Jensen%27s_inequality)).

The standard errors are the subtler casualty. Counts accumulated over varying exposures that the model doesn't know about vary more than a Poisson distribution allows: they're *overdispersed*. A Poisson model assumes `Var[y | X] = E[y | X]`, so it underestimates the uncertainty in its estimates, and its z-scores come out too large on average, though in any one fit, like this one, that can be hard to see. A `QuasiPoisson` model estimates the dispersion, so we can see it directly:

In [15]:
for label, offset in [("with the offset", sim.offset), ("without the offset", None)]:
    quasi_model = GLM(QuasiPoisson()).fit(sim.data, formula=sim.formula, offset=offset)
    print(f"Dispersion {label}: {quasi_model.dispersion_:.2f}")

Dispersion with the offset: 1.15
Dispersion without the offset: 1.45


With the offset, the estimate is near 1, as it should be for genuinely Poisson counts. It reads a little high, because `dispersion_` uses a deviance-based estimate, which runs high for counts this small. Without the offset it rises substantially: the exposures' variation shows up as overdispersion, and the Poisson model's standard errors are too small by roughly the square root of the increase. The notebook on inference returns to overdispersion, and to how bootstrap sampling may be used to detect it.

## Gamma and Exponential: positive amounts

For positive, continuous, right-skewed responses, like claim sizes or waiting times, the Gamma family with a log link is a common choice. Its variance grows with the square of its mean, `Var[y | X] = φ μ²`, so the *coefficient of variation*, `sd / mean = √φ`, is the same for every observation. This makes it a natural model for amounts whose noise is proportional to their size.

In [16]:
sim = make_frame(
    Gamma(),
    n_samples=2000,
    n_uncorrelated=3,
    n_correlated=3,
    coef_range=(-0.5, 0.5),
    dispersion=0.25,
    correlation=0.8,
    rng=np.random.default_rng(4),
)
gamma_model = GLM(Gamma()).fit(sim.data, formula=sim.formula)
compare(gamma_model, sim)

,estimate,std error,truth,z
Intercept,-0.450,0.012,-0.460,0.838
x1,-0.208,0.012,-0.210,0.104
x2,0.323,0.012,0.325,-0.139
x3,0.152,0.012,0.142,0.861
x4,-0.155,0.023,-0.103,-2.286
x5,-0.166,0.022,-0.201,1.587
x6,-0.261,0.022,-0.282,0.952


The Gamma's dispersion `φ` is estimated, here close to its true value of 0.25:

In [17]:
gamma_model.dispersion_

0.26787350596897325

The family of Exponential distributions (not to be confused with the more general concept of an Exponential Family of distributions, which all of our examples are) is the special case `φ = 1`: it has the same mean and variance functions, so it finds exactly the same coefficients, but it *assumes* the dispersion rather than estimating it. When that assumption is wrong, so are the standard errors. Here the true `φ` is 0.25, so the Exponential model's standard errors are about `√(1 / 0.25) = 2` times too large, and its z-scores about half what they should be:

In [18]:
exponential_model = GLM(Exponential()).fit(sim.data, formula=sim.formula)
compare(exponential_model, sim)

,estimate,std error,truth,z
Intercept,-0.450,0.022,-0.460,0.434
x1,-0.208,0.023,-0.210,0.054
x2,0.323,0.023,0.325,-0.072
x3,0.152,0.023,0.142,0.446
x4,-0.155,0.044,-0.103,-1.183
x5,-0.166,0.043,-0.201,0.821
x6,-0.261,0.043,-0.282,0.493


In [19]:
exponential_model.coef_standard_error_ / gamma_model.coef_standard_error_

array([1.93212455, 1.93212455, 1.93212455, 1.93212455, 1.93212455,
       1.93212455, 1.93212455])

Precisely, the ratio is `1 / √φ̂`, with `φ̂` the Gamma model's estimate: standard errors scale with the square root of the dispersion.

## Summary

| Family | Response | Link g | Variance Var[y \| X] | Dispersion |
|---|---|---|---|---|
| `Gaussian` | continuous | identity | φ | estimated (σ²) |
| `Bernoulli` | binary | logit | μ (1 - μ) | none (fixed at 1) |
| `Poisson` | count | log | μ | none (fixed at 1) |
| `QuasiPoisson` | count | log | φ μ | estimated |
| `Gamma` | positive | log | φ μ² | estimated |
| `Exponential` | positive | log | μ² | none (fixed at 1) |

`QuasiPoisson` is the Poisson model with an estimated dispersion, for counts more variable than a Poisson distribution allows; the notebook on inference returns to that, and to what happens when a model's variance assumption is wrong.